In [0]:
# Databricks notebook source
# COMMAND ----------
# MAGIC %md
# MAGIC ### Scalable Data Generator: 20,000 CPQ Payloads Per Batch
# MAGIC Generates Batch 1 (20k records with anomalies) and Batch 2 (20k clean records).

# COMMAND ----------
import json
import random
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, ArrayType
from pyspark.sql.functions import col, udf

DBFS_LANDING_PATH = "/Volumes/cpq_dev_ws/default/manufacturing_pipeline_files/raw_cpq_data/"
RECORDS_PER_BATCH = 20000

# Helper function to generate payloads
def generate_large_cpq_batch(batch_id, num_records, inject_anomalies=False):
    customers = ["Vertiv Corp", "Emerson Electric", "ABB Ltd", "Schneider Electric", "Siemens AG"]
    product_lines = [
        {"part": "PWR-UPS-5000", "base_cost": 3200.00, "cat": "Power Systems"},
        {"part": "COOL-INROW-01", "base_cost": 1800.00, "cat": "Thermal Management"},
        {"part": "RACK-ENCL-42U", "base_cost": 450.00, "cat": "Enclosures"},
        {"part": "SWITCH-PDU-32A", "base_cost": 210.00, "cat": "Power Distribution"}
    ]
    
    quotes = []
    base_date = datetime.now() - timedelta(days=30)
    
    for i in range(1, num_records + 1):
        quote_id = f"Q-2026-B{batch_id}-{100000 + i}"
        customer = customers[i % len(customers)]
        quote_date = (base_date + timedelta(seconds=(i * 100) % 2592000)).strftime("%Y-%m-%d %H:%M:%S")
        
        prod = product_lines[i % len(product_lines)]
        cost = prod["base_cost"]
        
        # Inject periodic anomalies (~0.5% rate in Batch 1)
        if inject_anomalies and (i % 200 == 0):
            if i % 600 == 0:
                net_price = -500.00  # Anomaly 1: Negative revenue
            elif i % 400 == 0:
                cost = 0.00          # Anomaly 2: Zero cost glitch
                net_price = 5000.00
            else:
                net_price = 180000.00 # Anomaly 3: 100x pricing typo
        else:
            # Clean discounting
            discount = random.choice([0.05, 0.10, 0.15, 0.20, 0.25])
            net_price = round((cost * 1.4) * (1 - discount), 2)
            
        line_items = [{
            "sequence_number": 1,
            "part_number": prod["part"],
            "product_category": prod["cat"],
            "unit_cost": cost,
            "list_price": round(cost * 1.4, 2),
            "net_price": net_price,
            "quantity": (i % 5) + 1,
            "bml_attributes_str": "region=EMEA;approval_required=FALSE"
        }]
            
        quotes.append({
            "transactionId": quote_id,
            "customerName": customer,
            "quoteDate": quote_date,
            "currency": "USD",
            "lineItems": line_items
        })
        
    return quotes

# COMMAND ----------
# 1. Generate & Write Batch 1 (20,000 Records with ~100 Anomalies)
print("⏳ Generating Batch 1 (20,000 Records)...")
batch1_data = generate_large_cpq_batch(batch_id=1, num_records=RECORDS_PER_BATCH, inject_anomalies=True)

# Write via PySpark DataFrame for fast parallel write
df_batch1 = spark.createDataFrame(batch1_data)
df_batch1.coalesce(1).write.mode("overwrite").json(f"{DBFS_LANDING_PATH}batch1_raw")

# COMMAND ----------
# 2. Generate & Write Batch 2 (20,000 Clean Records)
print("⏳ Generating Batch 2 (20,000 Records)...")
batch2_data = generate_large_cpq_batch(batch_id=2, num_records=RECORDS_PER_BATCH, inject_anomalies=False)

df_batch2 = spark.createDataFrame(batch2_data)
df_batch2.coalesce(1).write.mode("overwrite").json(f"{DBFS_LANDING_PATH}batch2_raw")

print(f"✅ Successfully generated two files with {RECORDS_PER_BATCH:,} records each in DBFS!")